# Signage & Wayfinding — Embedding Generator

This notebook:
1. Takes your comments CSV
2. Generates text embeddings via OpenAI
3. Pushes the result straight to your GitHub Pages site

**Run each cell top to bottom. You will be prompted for your API key, GitHub token, and CSV file.**

In [ ]:
# Install dependencies
!pip install openai numpy requests openpyxl -q

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
GITHUB_OWNER  = "tinydogboots"
GITHUB_REPO   = "apdxvc"
GITHUB_BRANCH = "gh-pages"

OUTPUT_NAME = "comments"
TENSOR_NAME = "Signage & Wayfinding Comments"

EMBED_MODEL   = "text-embedding-3-small"
ENRICH_MODEL  = "gpt-4o-mini"   # enriches each comment with conceptual meaning
RELATE_MODEL  = "gpt-4o"        # extracts relationships between comments
# ────────────────────────────────────────────────────────────────────────────────
print("Config set.")

In [ ]:
import getpass

OPENAI_KEY   = getpass.getpass("Paste your OpenAI API key: ").strip()
GITHUB_TOKEN = getpass.getpass("Paste your GitHub personal access token (needs repo scope): ").strip()

# Sanitize: keep only printable ASCII (removes hidden/mobile characters)
OPENAI_KEY   = "".join(c for c in OPENAI_KEY   if 32 <= ord(c) < 127)
GITHUB_TOKEN = "".join(c for c in GITHUB_TOKEN if 32 <= ord(c) < 127)

print(f"OpenAI key  : {OPENAI_KEY[:8]}...{OPENAI_KEY[-4:]} ({len(OPENAI_KEY)} chars)")
print(f"GitHub token: {GITHUB_TOKEN[:4]}...{GITHUB_TOKEN[-4:]} ({len(GITHUB_TOKEN)} chars)")
print("Credentials stored in memory only.")

**Need a GitHub token?**
1. Go to github.com → Settings → Developer settings → Personal access tokens → Tokens (classic)
2. Click **Generate new token (classic)**
3. Check the **repo** scope
4. Copy and paste it above

In [ ]:
# Upload your CSV or Excel file
from google.colab import files
import io, csv
import openpyxl

print("Select your comments file (CSV or Excel)...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
raw_bytes = uploaded[filename]

if filename.endswith(".xlsx") or filename.endswith(".xls"):
    wb = openpyxl.load_workbook(io.BytesIO(raw_bytes))
    ws = wb.active
    data = list(ws.values)
    columns = [str(c) for c in data[0]]
    rows = [dict(zip(columns, [str(v) if v is not None else "" for v in row])) for row in data[1:]]
    print(f"Loaded Excel: {filename}")
else:
    raw_bytes_data = raw_bytes
    for enc in ("utf-8-sig", "utf-8", "latin-1", "cp1252"):
        try:
            raw = raw_bytes_data.decode(enc)
            print(f"Loaded CSV: {filename} (encoding: {enc})")
            break
        except UnicodeDecodeError:
            continue
    reader = csv.DictReader(io.StringIO(raw))
    rows = list(reader)
    columns = reader.fieldnames

print(f"{len(rows)} rows, columns: {columns}")

In [ ]:
# ── Tell us which column holds the comment text ────────────────────────────────
COMMENT_COLUMN  = "comment"   # ← change to match your CSV/Excel header
TIER_COLUMN     = "tier"      # foundational / intermediate / advanced
CATEGORY_COLUMN = "category"  # signage and wayfinding / brand expression
# ────────────────────────────────────────────────────────────────────────────────

texts      = [r[COMMENT_COLUMN]  for r in rows]
tiers      = [r.get(TIER_COLUMN, "")     for r in rows]
categories = [r.get(CATEGORY_COLUMN, "") for r in rows]

print(f"First comment : {texts[0][:120]}")
print(f"Tier          : {tiers[0]}")
print(f"Category      : {categories[0]}")

In [ ]:
import json, time
from openai import OpenAI

client = OpenAI(api_key=OPENAI_KEY)

SYSTEM_PROMPT = """You are an expert in signage, wayfinding, and brand experience design.
You are analyzing feedback collected from employee travel-alongs, site audits, and digital surveys.

The feedback is organized using a Maslow-style pyramid framework with three tiers:
- Foundational: survival-level needs — basic navigation that people cannot function without
- Intermediate: quality improvements — things work but the experience could be much better
- Advanced: brand excellence — refinement, aspiration, emotional resonance

Each comment belongs to one of two categories:
- Signage & Wayfinding: physical navigation, signs, directories, maps, entry/exit clarity
- Brand Expression: visual identity, consistency, tone, environmental storytelling

Your job is to go beyond the literal words and understand the deeper meaning of each comment."""

def enrich_comment(comment, tier, category):
    prompt = f"""Analyze this comment from a {tier} / {category} context:

\"{comment}\"

Return a JSON object with these exact keys:
{{
  "core_problem": "one sentence — the actual underlying issue this reveals",
  "concept": "the specific design concept this touches (e.g. 'entry wayfinding hierarchy', 'brand touchpoint consistency')",
  "depends_on": "what must be resolved before this can be addressed (one phrase)",
  "enables": "what becomes possible once this is resolved (one phrase)",
  "enriched_text": "2-3 sentences combining the comment's literal meaning with its conceptual significance in the framework"
}}

Return only valid JSON, no other text."""

    resp = client.chat.completions.create(
        model=ENRICH_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.3,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

enriched = []
for i, (text, tier, cat) in enumerate(zip(texts, tiers, categories)):
    result = enrich_comment(text, tier, cat)
    result["original"] = text
    result["tier"]     = tier
    result["category"] = cat
    result["id"]       = str(i)
    enriched.append(result)
    if (i + 1) % 10 == 0 or i == len(texts) - 1:
        print(f"Enriched {i+1}/{len(texts)}")

print("\nSample enrichment:")
e = enriched[0]
print(f"  Original     : {e['original'][:80]}")
print(f"  Core problem : {e['core_problem']}")
print(f"  Concept      : {e['concept']}")
print(f"  Depends on   : {e['depends_on']}")
print(f"  Enables      : {e['enables']}")

In [ ]:
import numpy as np

# Embed the enriched_text (conceptually aware) instead of raw comments
embed_inputs = [e["enriched_text"] for e in enriched]

BATCH = 100
all_vectors = []

for i in range(0, len(embed_inputs), BATCH):
    batch = embed_inputs[i:i+BATCH]
    resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
    all_vectors.extend([d.embedding for d in resp.data])
    print(f"Embedded {min(i+BATCH, len(embed_inputs))}/{len(embed_inputs)}")

vectors = np.array(all_vectors, dtype=np.float32)
print(f"Vectors shape: {vectors.shape}")

In [ ]:
import base64, json

# Build .bytes content (raw float32, little-endian)
bytes_content = vectors.astype("<f4").tobytes()
bytes_b64 = base64.b64encode(bytes_content).decode()

# Build metadata TSV — include all enrichment fields for hover info
tsv_header = ["comment", "tier", "category", "core_problem", "concept", "depends_on", "enables"]
tsv_lines  = ["\t".join(tsv_header)]
for e in enriched:
    row = [
        e.get("original", ""),
        e.get("tier", ""),
        e.get("category", ""),
        e.get("core_problem", ""),
        e.get("concept", ""),
        e.get("depends_on", ""),
        e.get("enables", ""),
    ]
    tsv_lines.append("\t".join(str(v).replace("\t", " ").replace("\n", " ") for v in row))

tsv_content = "\n".join(tsv_lines)
tsv_b64 = base64.b64encode(tsv_content.encode()).decode()

print("Files ready.")
print(f"  .bytes : {len(bytes_content):,} bytes")
print(f"  .tsv   : {len(tsv_lines)-1} rows, columns: {tsv_header}")

In [ ]:
import requests, base64

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
}
API      = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents"
BASE_GIT = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/git"
BRANCHES = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/branches"

def get_sha(path):
    r = requests.get(f"{API}/{path}", headers=HEADERS, params={"ref": GITHUB_BRANCH})
    return r.json().get("sha") if r.status_code == 200 else None

def get_branch_head():
    r = requests.get(f"{BRANCHES}/{GITHUB_BRANCH}", headers=HEADERS)
    r.raise_for_status()
    b = r.json()
    return b["commit"]["sha"], b["commit"]["commit"]["tree"]["sha"]

def push_via_git_api(path, b64_content, message):
    """Use Git Data API — works for any file size."""
    # 1. Create blob
    r = requests.post(f"{BASE_GIT}/blobs", headers=HEADERS,
                      json={"content": b64_content, "encoding": "base64"})
    r.raise_for_status()
    blob_sha = r.json()["sha"]

    # 2. Get current branch head
    head_sha, base_tree = get_branch_head()

    # 3. Create tree
    r = requests.post(f"{BASE_GIT}/trees", headers=HEADERS, json={
        "base_tree": base_tree,
        "tree": [{"path": path, "mode": "100644", "type": "blob", "sha": blob_sha}]
    })
    r.raise_for_status()
    tree_sha = r.json()["sha"]

    # 4. Create commit
    r = requests.post(f"{BASE_GIT}/commits", headers=HEADERS, json={
        "message": message, "parents": [head_sha], "tree": tree_sha
    })
    r.raise_for_status()
    commit_sha = r.json()["sha"]

    # 5. Advance branch ref
    r = requests.patch(
        f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/git/refs/heads/{GITHUB_BRANCH}",
        headers=HEADERS, json={"sha": commit_sha}
    )
    r.raise_for_status()
    print(f"  ✓ {path}")

bytes_path = f"data/{OUTPUT_NAME}.bytes"
tsv_path   = f"data/{OUTPUT_NAME}_metadata.tsv"

print("Pushing to GitHub...")
push_via_git_api(bytes_path, bytes_b64, f"Add {OUTPUT_NAME} embeddings (.bytes)")
push_via_git_api(tsv_path,   tsv_b64,  f"Add {OUTPUT_NAME} metadata (.tsv)")
print("Done.")

In [ ]:
# Fetch current projector_config.json and add/replace this embedding
config_path = "data/projector_config.json"
url = f"{API}/{config_path}"
r = requests.get(url, headers={**HEADERS}, params={"ref": GITHUB_BRANCH})
r.raise_for_status()
config_data = r.json()
config = json.loads(base64.b64decode(config_data["content"]).decode())
sha = config_data["sha"]

entry = {
    "tensorName": TENSOR_NAME,
    "tensorShape": list(vectors.shape),
    "tensorPath": bytes_path,
    "metadataPath": tsv_path,
}
config["embeddings"] = [e for e in config["embeddings"] if e.get("tensorName") != TENSOR_NAME]
config["embeddings"].insert(0, entry)

new_b64 = base64.b64encode(json.dumps(config, indent=2).encode()).decode()
r = requests.put(url, headers=HEADERS, json={
    "message": f"Add {TENSOR_NAME} to projector config",
    "content": new_b64,
    "sha": sha,
    "branch": GITHUB_BRANCH,
})
r.raise_for_status()
print("Updated projector_config.json")
print(f"\n✅ All done! Open your projector at:")
print(f"   https://{GITHUB_OWNER}.github.io/{GITHUB_REPO}/")

In [ ]:
# ── Step 3: Extract explicit relationships between comments ────────────────────
# Build a numbered list of all comments for GPT-4o to reason over

comment_list = "\n".join(
    f"[{e['id']}] ({e['tier']} / {e['category']}) {e['original']}"
    for e in enriched
)

relate_prompt = f"""You are analyzing {len(enriched)} comments from a signage, wayfinding, and brand experience engagement.

The framework has three tiers (Maslow-style pyramid):
- Foundational: survival needs — basic navigation
- Intermediate: quality of experience
- Advanced: brand excellence / aspiration

Two categories per tier: Signage & Wayfinding | Brand Expression

Here are all the comments, numbered by ID:

{comment_list}

Identify the 60 most meaningful connections between these comments.
A connection exists when:
- One comment reveals a problem that must be solved before another can be addressed (dependency)
- Two comments describe different symptoms of the same root cause (shared root)
- Resolving one comment directly improves or enables another (enabler)
- Two comments are in tension or contradiction (tension)
- Two comments reinforce each other across different tiers or categories (cross-tier bridge)

Return a JSON object with a single key "connections", containing an array of objects:
{{
  "connections": [
    {{
      "from": "id of first comment",
      "to": "id of second comment",
      "type": "dependency | shared_root | enabler | tension | cross_tier_bridge",
      "label": "short phrase describing the relationship",
      "reason": "one sentence explaining why these are connected"
    }}
  ]
}}

Return only valid JSON."""

print("Extracting relationships with GPT-4o (this takes ~30 seconds)...")
resp = client.chat.completions.create(
    model=RELATE_MODEL,
    messages=[
        {"role": "system", "content": "You are an expert in spatial design, wayfinding, and brand experience strategy."},
        {"role": "user",   "content": relate_prompt}
    ],
    temperature=0.4,
    response_format={"type": "json_object"},
)
relationships = json.loads(resp.choices[0].message.content)["connections"]
print(f"Extracted {len(relationships)} relationships.")
print(f"\nSample:")
r = relationships[0]
print(f"  [{r['from']}] → [{r['to']}]  ({r['type']})")
print(f"  {r['label']}: {r['reason']}")

In [ ]:
# ── Step 4: Build and push network blueprint ───────────────────────────────────

elements = []
for e in enriched:
    elements.append({
        "id":    e["id"],
        "label": e["original"][:80] + ("..." if len(e["original"]) > 80 else ""),
        "type":  e["tier"],
        "description": e["enriched_text"],
        "tags":  [e["tier"], e["category"]],
        "attributes": {
            "tier":         e["tier"],
            "category":     e["category"],
            "core problem": e.get("core_problem", ""),
            "concept":      e.get("concept", ""),
            "depends on":   e.get("depends_on", ""),
            "enables":      e.get("enables", ""),
        }
    })

connections = []
for i, rel in enumerate(relationships):
    connections.append({
        "id":        f"conn_{i}",
        "from":      rel["from"],
        "to":        rel["to"],
        "label":     rel["label"],
        "direction": "directed",
        "attributes": {
            "type":   rel["type"],
            "reason": rel["reason"],
        }
    })

blueprint     = {"elements": elements, "connections": connections}
blueprint_b64 = base64.b64encode(json.dumps(blueprint, indent=2).encode()).decode()

print("Pushing network blueprint to GitHub...")
push_via_git_api("data/kumu_blueprint.json", blueprint_b64, "Update relationship blueprint")

print(f"\n✅ All done!")
print(f"   Embedding Projector : https://{GITHUB_OWNER}.github.io/{GITHUB_REPO}/")
print(f"   Relationship Map    : https://{GITHUB_OWNER}.github.io/{GITHUB_REPO}/network.html")